# Musanga V4 — NB01: spatial groups and cover-stratified RGB validation (v4_001)

**Purpose.** Create a new, explicitly versioned validation design comparing **A: original exhaustive circles + partial annotations** against **B: circles + exhaustive field plots + partial annotations**. Preserve historical V3.1 fold IDs as a separate secondary benchmark; **do not overwrite their CSVs or the existing V4 splits.**

NB00 v4_001 Section 09 must first create `Outputs/v4/00_reference_expansion_v1/00_reference_registry_108.gpkg` containing 30 original V3.1 circles and 78 V4 plots.

**Reference definition:** cover-stratification uses Nadine mapped Musanga **crown area / plot area** and the original V3.1 circle **Musanga cover / circle area**. It does **not** use field basal-area fraction. Mismatches in definitions must be examined before publication. Overlap is grouped, never independently split.

**Scientific control:** This makes new *development* folds. The V3.1 folds remain separate for historical comparison. Do not use the test-fold outcomes to select training hyperparameters. Performance must be compared on exactly the same held-out reference areas, with per-site and cover-class metrics.


## 00 — Load original references and audit the cover gradients

No raster reads, RGB extraction or GPU. A plot with zero intersection may still be near a circle; proximity is handled in the next section.


In [ ]:
%pip -q install geopandas pyogrio shapely pandas numpy matplotlib
from pathlib import Path
from collections import Counter
import json
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')] if p.exists()),None)
assert ROOT is not None
REG=ROOT/'Outputs/v4/00_reference_expansion_v1/00_reference_registry_108.gpkg'
assert REG.exists(),'Run revised NB00 v4_001, Section 09 first'
g=gpd.read_file(REG).to_crs('EPSG:32635')
assert len(g)==108 and (g.unit_type=='circle').sum()==30 and (g.unit_type=='plot').sum()==78
assert g.cover_pct.between(0,100).all()
assert set(g.site)=={'Baego','MOS','Yangambi','Yoko'}
OUT=ROOT/'Outputs/v4/01_spatial_cover_folds_v1';OUT.mkdir(parents=True,exist_ok=True)
BINS=[-1e-9,5,20,40,100.000001]
LABELS=['0-5','5-20','20-40','40-100']
g['cover_bin']=pd.cut(g.cover_pct,bins=BINS,labels=LABELS,include_lowest=True,right=True).astype(str)
print('Counts by site / type');display(pd.crosstab(g.site,g.unit_type))
print('Cover classes by site / type')
display(pd.crosstab([g.site,g.unit_type],g.cover_bin).reindex(columns=LABELS,fill_value=0))
print('Percent cover summary by type')
display(g.groupby(['site','unit_type']).cover_pct.describe().round(2))
fig,axs=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for kind,grp in g.groupby('unit_type'):
    axs[0].hist(grp.cover_pct,bins=np.arange(0,105,5),alpha=.5,label=f'{kind} (n={len(grp)})')
axs[0].set(xlabel='Reference Musanga canopy cover (%)',ylabel='Reference units',title='Cover distribution')
axs[0].legend()
for site,grp in g.groupby('site'):
    axs[1].scatter([site]*len(grp),grp.cover_pct,alpha=.7,s=25,label=site)
axs[1].set(ylabel='Reference cover (%)',title='Site gradient')
plt.show()


## 01 — Group overlapping or nearby units BEFORE choosing validation folds

All spatially connected reference footprints become one group. A **15 m** initial proximity threshold is retained for comparability with V3.1, but train-chip exclusion must subsequently account for full crop footprints. Plot-in-circle observations stay linked and must not be treated as independent error replicates. Group membership is written to CSV for direct inspection.


In [ ]:
from shapely.strtree import STRtree
N=len(g);DISTANCE_M=15.0
geoms=list(g.geometry)
tree=STRtree(geoms)
parent=np.arange(N)
def find(i):
    i=int(i)
    while parent[i]!=i:
        parent[i]=parent[parent[i]];i=int(parent[i])
    return i
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb:parent[max(ra,rb)]=min(ra,rb)
overlap_pairs=[];near_pairs=[]
for i,geom in enumerate(geoms):
    for raw_j in tree.query(geom.buffer(DISTANCE_M),predicate='intersects'):
        j=int(raw_j)
        if j<=i:continue
        dist=float(geom.distance(geoms[j]))
        if dist<=DISTANCE_M+1e-8:
            assert g.site.iloc[i]==g.site.iloc[j],('Cross-site overlap',i,j)
            union(i,j)
            (overlap_pairs if dist<=1e-8 else near_pairs).append((i,j,dist))
roots=[find(i) for i in range(N)]
rootmap={r:k for k,r in enumerate(sorted(set(roots)))}
g['spatial_group']=['SG%03d'%rootmap[r] for r in roots]
print('Spatial groups:',g.spatial_group.nunique(),'for',N,'reference units')
print('Overlapping/touching pairs:',len(overlap_pairs),'Within 15 m but not intersecting:',len(near_pairs))
print('Groups of multiple references:')
display(g.groupby(['spatial_group','site']).agg(
 n=('unit_id','size'),n_circles=('unit_type',lambda x:int((x=='circle').sum())),
 n_plots=('unit_type',lambda x:int((x=='plot').sum())),
 cover_min=('cover_pct','min'),cover_max=('cover_pct','max')
).query('n>1').sort_values('n',ascending=False).round(2))
print('Site / group counts:')
display(g[['site','spatial_group']].drop_duplicates().groupby('site').size().to_frame('independent_groups'))


## 02 — Optimize three outer folds for cover and site balance, without breaking spatial groups

This creates a deterministic **candidate** new split: repeated group-level randomized greedy assignment, selecting the arrangement minimizing site-by-cover, site-level and unit-count imbalance. Exact equality is not assumed possible. Outputs include an explicit feasibility check; review the diagnostics before freezing and training. Historical V3.1 fold IDs remain unchanged.


In [ ]:
from collections import defaultdict
K=3
group_info=[]
for gid,q in g.groupby('spatial_group'):
    site=str(q.site.iloc[0]);assert q.site.nunique()==1
    v=np.array([(q.cover_bin.eq(b)).sum() for b in LABELS],dtype=float)
    group_info.append(dict(gid=gid,site=site,n=len(q),bins=v,
      n_circle=int(q.unit_type.eq('circle').sum()),
      n_plot=int(q.unit_type.eq('plot').sum())))
sites=sorted(g.site.unique())
site_ix={s:i for i,s in enumerate(sites)}
total=np.zeros((len(sites),4))
for z in group_info:total[site_ix[z['site']]]+=z['bins']
def score(a):
    # Penalize site×cover imbalance but downweight very sparse strata; control total units.
    counts=np.zeros((K,len(sites),4));sizes=np.zeros(K);gcounts=np.zeros((K,len(sites)))
    for z in group_info:
        f=a[z['gid']];counts[f,site_ix[z['site']]]+=z['bins']
        sizes[f]+=z['n'];gcounts[f,site_ix[z['site']]]+=1
    group_target=np.array([(sum(z['site']==s for z in group_info))/K for s in sites])
    cost=np.sum((counts-total[None]/K)**2/np.maximum(total[None]/K,1.5))
    cost+=.6*np.sum((gcounts-group_target[None])**2/np.maximum(group_target[None],1))
    cost+=.12*np.sum((sizes-len(g)/K)**2/(len(g)/K))
    # Empty site-fold groups are highly undesirable if feasible.
    cost+=200*np.sum((gcounts==0)&(group_target[None]>0))
    return float(cost)
best=None;bestscore=float('inf')
for trial in range(500):
    rng=np.random.default_rng(20261118+trial)
    ordered=sorted(group_info,key=lambda z:(-z['n'],rng.random()))
    assign={}
    for z in ordered:
        costs=[]
        for f in range(K):
            assign[z['gid']]=f
            # Score partial assignment using a fixed eventual target; favors balanced progress.
            costs.append(score(assign) if len(assign)==len(group_info) else None)
        # For partial assignment use incremental occupancy penalty.
        if len(assign)<len(group_info):
            partial=np.zeros((K,len(sites),4))
            gs=np.zeros((K,len(sites)))
            for q in group_info:
                if q['gid'] in assign and q['gid']!=z['gid']:
                    ff=assign[q['gid']];ss=site_ix[q['site']]
                    partial[ff,ss]+=q['bins'];gs[ff,ss]+=1
            costs=[]
            for f in range(K):
                candidate=partial.copy();candidate[f,site_ix[z['site']]]+=z['bins']
                cell=np.maximum(total[None]/K,1.5)
                costs.append(float(np.sum(candidate*candidate/cell))+
                      .25*float(np.sum((gs+np.eye(K,len(sites),k=site_ix[z['site']])[f][None] if False else gs)**2)))
        chosen=int(np.argmin(np.array(costs)+rng.random(K)*1e-7))
        assign[z['gid']]=chosen
    # Optimize actual global objective by single-group moves.
    for iteration in range(8):
        changed=False
        for z in rng.permutation(group_info):
            gid=z['gid'];cur=assign[gid];base=score(assign)
            for f in range(K):
                if f==cur:continue
                assign[gid]=f;v=score(assign)
                if v+1e-9<base:cur=f;base=v;changed=True
            assign[gid]=cur
        if not changed:break
    v=score(assign)
    if v<bestscore:bestscore=v;best=dict(assign)
g['fold']=g.spatial_group.map(best).astype(int)
assert g.groupby('spatial_group').fold.nunique().eq(1).all()
print('Balanced split objective:',round(bestscore,3))
print('Site × fold × cover class reference units')
display(pd.crosstab([g.site,g.fold],g.cover_bin).reindex(columns=LABELS,fill_value=0))
print('Site × fold × unit type')
display(pd.crosstab([g.site,g.fold],g.unit_type))
print('Site × fold × independent groups')
display(g[['site','fold','spatial_group']].drop_duplicates().groupby(['site','fold']).size().unstack(fill_value=0))
print('Site × fold cover summary')
display(g.groupby(['site','fold']).cover_pct.agg(['count','min','median','max']).round(2))
unrepresented=[(s,f) for s in sites for f in range(K) if not ((g.site==s)&(g.fold==f)).any()]
if unrepresented:raise RuntimeError('UNACCEPTABLE empty site folds: '+str(unrepresented))
# Do not assert each cover class exists within every fold when not geographically feasible.


## 03 — Freeze the candidate reference and split manifests

**Important:** A frozen group-level fold manifest does not itself guarantee no chip leakage. The future revised NB02 must reject every training window intersecting a held-out group's full footprint plus a spatial buffer. Report historical folds separately; they cannot be assigned to new V4 plots without a new explicit overlap check.


In [ ]:
import hashlib
out=g[['unit_type','unit_id','site','cover_pct','cover_bin',
       'spatial_group','fold','historical_fold','reference_source','geometry']].copy()
out=out.sort_values(['site','spatial_group','unit_type','unit_id']).reset_index(drop=True)
p_csv=OUT/'01_reference_groups_and_folds_v1.csv'
p_gpkg=OUT/'01_reference_groups_and_folds_v1.gpkg'
out.drop(columns='geometry').to_csv(p_csv,index=False)
out.to_file(p_gpkg,driver='GPKG')
checksum=hashlib.sha256(p_csv.read_bytes()).hexdigest()
manifest=dict(schema='musanga_v4_01_cover_stratified_spatial_groups_v1',
   purpose='paired_rgb_circles_only_vs_circles_plus_plots',
   reference_registry=str(REG),n_circles=30,n_plots=78,n_total=108,
   spatial_group_proximity_m=DISTANCE_M,outer_folds=K,
   cover_bins_pct=BINS,cover_source_plot='Nadine_crown_union_area_over_supplied_plot_area',
   cover_source_circle='V3_1_KVO_final_exhaustive_positive_area_over_circle_area',
   historical_v31_folds='preserved_as_separate_secondary_benchmark',
   train_chip_exclusion='MUST exclude any held-out footprint plus spatial margin from full crop footprints',
   paired_evaluation='identical held-out reference units and valid RGB pixels for both RGB models',
   note='New cover-balanced spatial folds are development CV, not untouched independent test',
   output_csv=str(p_csv),csv_sha256=checksum)
(OUT/'01_reference_fold_contract_v1.json').write_text(json.dumps(manifest,indent=2))
print('WROTE:',p_csv,'SHA256:',checksum[:20])
print('WROTE:',p_gpkg)
print('WROTE:',OUT/'01_reference_fold_contract_v1.json')
print('NEXT: inspect folds, then update NB02 training-cache generation. No models trained.')
